# Calculator Model Training and SHAP/FFA Workflow

**Purpose:** Train calculator models and run SHAP + Formal Feature Attribution (FFA) analysis  
**Updated:** January 27, 2026  
**Hardware:** Optimized for EC2 instances  
**Model Strategy:** Dual model implementation (Baseline + Extended) for all cohorts

## Overview

This notebook provides an interactive workflow for:

1. **Training Calculator Models** - Train **two model variants** (Baseline and Extended) using the **Combined** cohort (single model for all patients)
2. **SHAP + FFA Analysis** - Generate causal factors and dashboard data using SHAP values and XGBoost rule extraction for both models
3. **Results Inspection** - View top causal factors, feature importance, and model performance
4. **Dashboard Deployment** - Deploy both models to the risk calculator dashboard with separate tabs

## Model Architecture

- **Dual Model Approach**: Two Combined models are trained for all cohorts (CHD, Cardiomyopathy, Myocarditis):
  - **Baseline Model** (`Combined_base`): Uses base calculator features only
  - **Extended Model** (`Combined_enhanced`): Uses base features + recommended additional features
- **Primary Diagnosis Feature**: `primary_etiology` is included to distinguish between etiologies
- **Feature Engineering**: Automatic derivation of combined variables (VAD, Ventilation, ECMO, donor ratios)

## Model Training Strategy

**Two Model Variants:**
1. **Baseline Model** (`Combined_base`): Base calculator features only (~104 features)
2. **Extended Model** (`Combined_enhanced`): Base features + recommended features (~120 features)

**For Each Model Variant, Three Model Types Trained:**
1. **CatBoost** - Gradient boosting with categorical feature support (Cox regression)
2. **XGBoost** - Extreme gradient boosting (Cox regression)
3. **XGBoost Random Forest** - XGBoost in Random Forest mode (Cox regression)

**Model Selection:**
- All three model types are trained on the same training data for each variant
- Performance is evaluated using C-index (Concordance Index)
- The model with the **highest C-index** is selected as the best model for each variant
- Both baseline and extended models are saved and deployed to the dashboard

**Dashboard Deployment:**
- **Baseline Model Tab**: Uses models from `Combined_base/` directory
- **Extended Model Tab**: Uses models from `Combined_enhanced/` directory
- Users can compare predictions from both models side-by-side

**Note:** This dual model approach allows users to choose between a simpler baseline model and a more comprehensive extended model with additional clinical features.

## Causal Analysis Strategy (SHAP + FFA)

**Important:** The causal analysis workflow uses a specific combination of models and applies rules to the **test set** for final causal analysis.

### Workflow Overview

```mermaid
graph TD
    A[Training Data] --> B[Temporal Split<br/>80/20]
    B --> C[Train Set<br/>txpl_year ≤ cutoff]
    B --> D[Test Set<br/>txpl_year > cutoff]
    
    C --> E[Train Models<br/>CatBoost, XGBoost, XGBoost RF]
    E --> F[Select Best Model<br/>by C-index, then AU-PRC]
    F --> G[Final Model<br/>Trained on Train Set]
    
    G --> H[Extract Rules<br/>from XGBoost JSON]
    D --> I[Compute SHAP Values<br/>on Test Set Only]
    
    H --> J[Apply Rules to Test Set<br/>Count Rule Firings]
    I --> K[Combine SHAP Values<br/>XGBoost + CatBoost if needed]
    
    J --> L[Calculate Rule Frequencies<br/>from Test Set]
    K --> M[SHAP Importance<br/>per Feature]
    
    L --> N[Causal Responsibility<br/>rule_freq × SHAP_importance]
    M --> N
    
    N --> O[Top K Causal Factors<br/>for Dashboard]
    
    style D fill:#e1f5ff
    style I fill:#e1f5ff
    style J fill:#e1f5ff
    style L fill:#e1f5ff
    style O fill:#c8e6c9
```

### Key Principles

**1. Test Set Application (Critical)**
- ✅ **Rules are extracted from the trained model** (trained on training set)
- ✅ **Rules are applied to the test set** (unseen data) for final causal analysis
- ✅ **SHAP values are computed on the test set only** (not training set)
- ✅ **Rule frequencies are counted from test set rule firings** (not from rule definitions)
- ✅ **Temporal split cutoff matches training** (dynamic 80/20 split, falls back to 2021)

**Why Test Set?**
- Ensures causal factors reflect model behavior on **unseen data**
- Prevents overfitting to training patterns
- Provides realistic causal responsibility scores
- Matches model evaluation methodology

### SHAP Values (Feature Importance)
- **Best XGBoost Model**: SHAP values are **always** computed from the best XGBoost model
- **Best CatBoost Model**: SHAP values are computed from the best CatBoost model **only if CatBoost is the best model**
- **Combination**: If CatBoost is best, SHAP values are combined with **auto-determined weights** based on C-index values
- **If XGBoost is best**: Only XGBoost SHAP values are used
- **Data Source**: SHAP values computed on **test set only** (`txpl_year > cutoff_year`)

### FFA Analysis (Rule Extraction)
- **Best XGBoost JSON Model**: Rules are **always** extracted from the best XGBoost JSON model
  - This is because XGBoost JSON structure is easier to parse for rule extraction
  - CatBoost JSON is not used (harder to parse due to categorical hashing)
- **Rule Source**: Rules extracted from model trained on **training set**
- **Rule Application**: Rules are **applied to test set instances** to count actual rule firings
- **Rule Filtering**: Rules are filtered using SHAP importance values (from step above)
- **Causal Responsibility**: Calculated as `(rule_frequency_from_test_set / total_rule_firings) × SHAP_importance`
  - `rule_frequency_from_test_set`: Count of how many times a rule fires on test set instances
  - `SHAP_importance`: Feature importance from SHAP values (computed on test set)

### Summary
**For causal analysis:**
1. ✅ SHAP values from **best XGBoost model** (always, computed on **test set**)
2. ✅ SHAP values from **best CatBoost model** (only if CatBoost is best, computed on **test set**)
3. ✅ Rules extracted from **best XGBoost JSON model** (trained on **training set**)
4. ✅ Rules applied to **test set** to count actual rule firings
5. ✅ FFA analysis combines **test set rule frequencies** + **test set SHAP values** to calculate causal responsibility

## Workflow Steps

- **Step 1:** Train Baseline Combined model (base calculator features)
- **Step 2:** Train Extended Combined model (base + recommended features)
- **Step 3:** Run SHAP/FFA analysis for Baseline model
- **Step 4:** Run SHAP/FFA analysis for Extended model
- **Step 5:** Inspect results and export dashboard data for both models
- **Step 6:** Deploy both models to dashboard (Baseline and Extended tabs)

## Expected Runtime

- **Baseline Model Training:** ~15-30 minutes
- **Extended Model Training:** ~15-30 minutes (with parallel processing)
- **SHAP/FFA Analysis (Baseline):** ~10-20 minutes
- **SHAP/FFA Analysis (Extended):** ~10-20 minutes
- **Total:** ~50-100 minutes on EC2 (can be parallelized)


## 1. Input Features Overview

### Required Input Variables for Risk Calculator

The model uses the following input features, with automatic feature engineering for derived variables:

#### Primary Diagnosis & History
- **Primary Diagnosis** (`primary_etiology`) - Congenital Heart Disease, Cardiomyopathy, Myocarditis, Other
- **Previous Cardiac Surgery** (`hxsurg`) - History of surgery (Yes/No)
- **Laterality Disorder** (`chd_lat`) - Composite variable (Yes/No)
  - Derived from: `chd_dex`, `chd_si`, `chd_heter`, `chd_iivc`, `chd_bivc`, `chd_lsvc`, `chd_raa`, `chd_avd`

#### Cardiac Support Devices (Combined Variables)
- **ECMO** (`ecmo_combined`) - ECMO at transplant OR listing
  - Derived from: `txecmo` OR `slecmo`
- **VAD** (`vad_combined`) - VAD at transplant OR listing
  - Derived from: `txvad` OR `slvad`
- **Mechanical Ventilation** (`vent_combined`) - Ventilation at transplant OR listing
  - Derived from: `txvent` OR `slvent` OR `ltxtrach` OR `hxtrach`

#### Demographics & Age
- **Age at Transplant** (`age_txpl`) - Years (priority over `age_listing`)
- **Age at Listing** (`age_listing`) - Years (fallback)

#### Renal Function
- **Dialysis History** (`hxdysdia` / `hxdysdia_bin`) - History of dialysis (ever)
- **eGFR at Transplant** (`egfr_tx`) - Calculated from height and creatinine
  - Formula: `egfr_tx = 0.413 × height_txpl / txcreat_r`
- **eGFR at Listing** (`egfr_listing`) - Calculated from height and creatinine

#### Liver Function
- **ALT at Transplant** (`txalt`) - U/L (priority over `lsalt`)
- **AST at Transplant** (`txast`) - U/L (priority over `lsast`)
- **Direct Bilirubin at Transplant** (`txbili_d_r`) - mg/dL (priority over `lsbili_d_r`)
- **Total Bilirubin at Transplant** (`txbili_t_r`) - mg/dL (priority over `lsbili_t_r`)

#### Nutrition
- **Serum Albumin at Transplant** (`txsa_r`) - g/dL (priority over `lssab_r`)
- **Total Protein at Transplant** (`txtp_r`) - g/dL (priority over `lstp_r`)

#### Immunology
- **cPRA at Transplant** (`txfcpra`) - Flow cytometry PRA % (priority over `lsfcpra`)
- **cPRA at Listing** (`lsfcpra`) - Flow cytometry PRA % (fallback)

#### Donor Characteristics
- **Donor Ischemic Time** (`donisch`) - Minutes (default: < 240 minutes if not provided)
- **Donor/Recipient Weight Ratio** (`donor_weight_ratio`) - Percentage
  - Formula: `(weight_donor / weight_txpl) × 100`
  - Model assumption: 70-200%
- **Donor/Recipient Size Ratio** (`donor_size_ratio`) - Percentage
  - Formula: `(height_donor / height_txpl) × 100`
  - Model assumption: 70-200%

### Additional Features

The model also includes:
- All CHD subtype variables (40+ subtypes, e.g., `chd_hlh`, `chd_lsvc`, `chd_si`, etc.)
- Additional lab values and clinical history variables
- Derived categorical variables (eGFR categories, high/low indicators)
- Donor characteristics and transplant details

### Feature Engineering

The following variables are automatically created during training and inference:
1. `ecmo_combined` - ECMO combined
2. `vad_combined` - VAD combined
3. `vent_combined` - Ventilation combined
4. `donor_weight_ratio` - Donor/recipient weight ratio
5. `donor_size_ratio` - Donor/recipient height ratio
6. `chd_lat` - Laterality disorder composite
7. `egfr_tx` - eGFR at transplant (if not provided, calculated from height/creatinine)
8. `egfr_listing` - eGFR at listing
9. `egfr_tx_cat` - eGFR category (severe/moderate/mild/normal)
10. `egfr_listing_cat` - eGFR category at listing
11. Additional derived variables (BMI, high/low indicators, etc.)

---

## 2. Setup and Configuration

Load required packages and configure paths.

In [2]:
import sys
from pathlib import Path
import logging
import warnings
warnings.filterwarnings('ignore')

# Add project paths
PROJECT_ROOT = Path().resolve().parent.parent.parent
CALCULATOR_DIR = Path().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
sys.path.insert(0, str(CALCULATOR_DIR))

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("=" * 80)
print("PHTS Calculator Workflow")
print("=" * 80)
print(f"Project root: {PROJECT_ROOT}")
print(f"Calculator directory: {CALCULATOR_DIR}")
print("=" * 80)

PHTS Calculator Workflow
Project root: /home/pgx3874/phts
Calculator directory: /home/pgx3874/phts/graft-loss/cohort_analysis/calculator


In [3]:
# Check Docker service
import subprocess
import platform

def check_docker():
    """Simple Docker check - verify if Docker is accessible."""
    print("\n" + "=" * 80)
    print("Docker Check")
    print("=" * 80)
    
    try:
        result = subprocess.run(
            ["docker", "ps"],
            capture_output=True,
            text=True,
            timeout=5
        )
        
        if result.returncode == 0:
            print("✓ Docker is running")
            logger.info("Docker is accessible")
            return True
        else:
            print("⚠ Docker is not accessible")
            if "permission denied" in result.stderr.lower():
                print("  Permission issue - you may need to add user to docker group:")
                print("    Linux: sudo usermod -aG docker $USER && newgrp docker")
            else:
                print(f"  Error: {result.stderr.strip()}")
            return False
            
    except FileNotFoundError:
        print("✗ Docker not found - please install Docker")
        system = platform.system()
        if system == "Windows":
            print("  Install Docker Desktop from: https://www.docker.com/products/docker-desktop")
        else:
            print("  Linux: https://docs.docker.com/engine/install/")
            print("  macOS: Install Docker Desktop")
        return False
    except Exception as e:
        print(f"⚠ Error checking Docker: {e}")
        return False
    
    print("=" * 80)

# Run check
docker_ok = check_docker()

2026-02-03 16:36:37,656 - __main__ - INFO - Docker is accessible



Docker Check
✓ Docker is running


In [4]:
# Timing helper for workflow steps (aligned with mermaid chart workflow)
import time
from contextlib import contextmanager

@contextmanager
def step_timer(step_name, sub_steps=None):
    """
    Context manager to time workflow steps with logging.
    
    Aligns with mermaid chart workflow:
    - Training: Temporal Split → Train Models → Select Best Model → Final Model
    - SHAP/FFA: Extract Rules → Compute SHAP → Apply Rules → Calculate Frequencies → Causal Responsibility
    
    Args:
        step_name: Main step name (e.g., "Step 1: Train Baseline Model")
        sub_steps: Optional list of sub-steps that align with mermaid chart nodes
    """
    start_time = time.time()
    start_str = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(start_time))
    print(f"\n{'=' * 80}")
    print(f"⏱️  START: {step_name}")
    print(f"   Started at: {start_str}")
    if sub_steps:
        print(f"   Sub-steps (per mermaid chart):")
        for i, sub_step in enumerate(sub_steps, 1):
            print(f"     {i}. {sub_step}")
    print(f"{'=' * 80}")
    logger.info(f"START: {step_name} at {start_str}")
    if sub_steps:
        logger.info(f"Sub-steps: {', '.join(sub_steps)}")
    
    try:
        yield
    finally:
        end_time = time.time()
        end_str = time.strftime("%Y-%m-%d %H:%M:%S", time.localtime(end_time))
        duration = end_time - start_time
        duration_min = duration / 60
        duration_sec = duration % 60
        
        print(f"\n{'=' * 80}")
        print(f"✅ COMPLETE: {step_name}")
        print(f"   Started: {start_str}")
        print(f"   Finished: {end_str}")
        print(f"   Duration: {duration_min:.1f} minutes ({duration:.0f} seconds)")
        print(f"{'=' * 80}")
        logger.info(f"COMPLETE: {step_name} - Duration: {duration_min:.1f} minutes ({duration:.0f} seconds)")

print("✓ Timing helper loaded (aligned with mermaid chart workflow)")

✓ Timing helper loaded (aligned with mermaid chart workflow)


In [5]:
# Configuration
DEBUG_MODE = False  # Set to True for quick testing (fewer splits)

# Model Strategy: Single Combined model for all cohorts
# Note: The training script will always train Combined model regardless of --cohort argument
COHORT = "Combined"  # Single model approach - Combined model for all cohorts

# SHAP/FFA configuration
TOP_K = 10  # Number of top causal factors to extract
# Note: Weights are automatically determined from best model C-index values
# Set to None to use auto-determination, or override manually if needed
WEIGHT_CATBOOST = None  # Auto-determined from best model (None = auto)
WEIGHT_XGBOOST = None   # Auto-determined from best model (None = auto)

print(f"\nConfiguration:")
print(f"  DEBUG_MODE: {DEBUG_MODE}")
print(f"  Model Strategy: Single Combined model (for all cohorts)")
print(f"  Cohort: {COHORT}")
print(f"  Top K factors: {TOP_K}")
print(f"  SHAP Weights: Auto-determined from best model C-index values")
print(f"    (CatBoost weight: {'Auto' if WEIGHT_CATBOOST is None else WEIGHT_CATBOOST})")
print(f"    (XGBoost weight: {'Auto' if WEIGHT_XGBOOST is None else WEIGHT_XGBOOST})")
print(f"\nNote: The model includes primary_etiology to distinguish between:")
print(f"  - Congenital Heart Disease")
print(f"  - Cardiomyopathy")
print(f"  - Myocarditis")
print(f"  - Other")


Configuration:
  DEBUG_MODE: False
  Model Strategy: Single Combined model (for all cohorts)
  Cohort: Combined
  Top K factors: 10
  SHAP Weights: Auto-determined from best model C-index values
    (CatBoost weight: Auto)
    (XGBoost weight: Auto)

Note: The model includes primary_etiology to distinguish between:
  - Congenital Heart Disease
  - Cardiomyopathy
  - Myocarditis
  - Other


In [6]:
# Check dependencies
print("\nChecking dependencies...")

try:
    import numpy as np
    import pandas as pd
    from catboost import CatBoostRegressor
    import xgboost as xgb
    import shap
    print("✓ All required packages are installed")
    print(f"  NumPy: {np.__version__}")
    print(f"  Pandas: {pd.__version__}")
    print(f"  XGBoost: {xgb.__version__}")
    print(f"  SHAP: {shap.__version__}")
except ImportError as e:
    print(f"✗ Missing dependency: {e}")
    print("  Please install: pip install numpy pandas catboost xgboost shap")


Checking dependencies...
✓ All required packages are installed
  NumPy: 1.26.4
  Pandas: 2.2.3
  XGBoost: 3.1.2
  SHAP: 0.47.2


In [7]:
# Check data availability
print("\nChecking data availability...")

data_file = PROJECT_ROOT / "graft-loss" / "data" / "phts_txpl_ml.sas7bdat"
if data_file.exists():
    size_mb = data_file.stat().st_size / (1024 * 1024)
    print(f"✓ Data file found: {data_file}")
    print(f"  Size: {size_mb:.2f} MB")
else:
    print(f"⚠ Data file not found: {data_file}")
    print("  You may need to download the data file first")

# Check calculator directory structure
outputs_dir = CALCULATOR_DIR / "outputs"
if outputs_dir.exists():
    print(f"✓ Outputs directory exists: {outputs_dir}")
else:
    print(f"✓ Creating outputs directory: {outputs_dir}")
    outputs_dir.mkdir(parents=True, exist_ok=True)


Checking data availability...
✓ Data file found: /home/pgx3874/phts/graft-loss/data/phts_txpl_ml.sas7bdat
  Size: 25.50 MB
✓ Outputs directory exists: /home/pgx3874/phts/graft-loss/cohort_analysis/calculator/outputs


## 3. Train Calculator Models

Train **all three model types** (CatBoost, XGBoost, and XGBoost RF) using the **Combined** cohort.

**Training Process:**
1. **Data Split**: Temporal 80/20 split (train on earlier years, test on later years)
2. **Model Training**: All three models are trained on the same training data:
   - CatBoost (Cox regression)
   - XGBoost (Cox regression)
   - XGBoost Random Forest (Cox regression)
3. **Model Evaluation**: C-index is calculated for each model on the test set
4. **Model Selection**: The model with the highest C-index is selected as the best model

**Note:** The training script enforces a single Combined model strategy. Even if you specify a different cohort, it will train the Combined model for all patients.

### a. Baseline Model

In [8]:
# Import training function
from train_python_models import train_models_for_cohort
import os
import multiprocessing

# Determine number of parallel jobs (use all available CPUs minus 1 for safety)
n_parallel_jobs = max(1, multiprocessing.cpu_count() - 1)

print(f"\n{'=' * 80}")
print("Training Baseline Calculator Model (Base Features Only)")
print(f"{'=' * 80}")
print(f"\nConfiguration:")
print(f"  Cohort: {COHORT}")
print(f"  Feature Set: Baseline (base calculator features only)")
print(f"  Parallel Jobs: {n_parallel_jobs} (using {multiprocessing.cpu_count()} CPUs)")
print(f"  MC-CV Splits: 25")
print(f"  Training Proportion: 80%")
print(f"\nTraining Process:")
print(f"  1. Monte Carlo Cross-Validation (25 splits)")
print(f"  2. Model Training: All three model types (CatBoost, XGBoost, XGBoost RF)")
print(f"  3. Model Selection: Best model by C-index, then AU-PRC")
print(f"  4. Final Model: Best model trained on full temporal split")
print("-" * 80)
print("\nModel Features:")
print("  - primary_etiology feature to distinguish etiologies")
print("  - All derived variables (vad_combined, vent_combined, donor ratios, chd_lat)")
print("  - All required input features from risk calculator")
print("-" * 80)

print(f"\nTraining Baseline Combined model (for all cohorts)...")
print(f"  Output directory: outputs/models/Combined_base/")

try:
    import time
    start_time = time.time()
    
    train_models_for_cohort(
        cohort=COHORT,
        n_mc_splits=25,
        train_prop=0.8,
        n_jobs=n_parallel_jobs,  # Use parallel processing
        include_recommended_features=False  # Baseline model uses base features only
    )
    
    elapsed_time = time.time() - start_time
    print(f"\n✓ Baseline Combined model training complete!")
    print(f"  Total time: {elapsed_time/60:.1f} minutes ({elapsed_time:.0f} seconds)")
    print(f"  Output saved to: outputs/models/Combined_base/")
    print(f"\nAll three models have been trained:")
    print(f"  - CatBoost")
    print(f"  - XGBoost")
    print(f"  - XGBoost Random Forest")
    print(f"\nThe best model (highest C-index) has been selected and saved.")
    print(f"\nThe model is now ready for:")
    print(f"  - Risk prediction for all cohorts (CHD, Cardiomyopathy, Myocarditis)")
    print(f"  - SHAP/FFA analysis to extract causal factors")
except Exception as e:
    print(f"\n✗ Error training Baseline Combined model: {e}")
    import traceback
    traceback.print_exc()

print(f"\n{'=' * 80}")
print("Baseline Model Training Complete!")
print(f"{'=' * 80}")

2026-02-03 16:37:12,018 - botocore.credentials - INFO - Found credentials from IAM Role: EC2_Spot
2026-02-03 16:37:12,206 - train_python_models - INFO - 
2026-02-03 16:37:12,206 - train_python_models - INFO - Training models for cohort: Combined
2026-02-03 16:37:12,207 - train_python_models - INFO - ================================================================================
2026-02-03 16:37:12,207 - train_python_models - INFO - MC-CV Configuration:
2026-02-03 16:37:12,207 - train_python_models - INFO -   - Number of splits: 25
2026-02-03 16:37:12,207 - train_python_models - INFO -   - Training proportion: 80.0%
2026-02-03 16:37:12,208 - train_python_models - INFO -   - Parallel jobs: 31
2026-02-03 16:37:12,208 - train_python_models - INFO -   - Time horizon for AUC/AU-PRC/Recall: 365.25 days
2026-02-03 16:37:12,208 - train_python_models - INFO - 



Training Baseline Calculator Model (Base Features Only)

Configuration:
  Cohort: Combined
  Feature Set: Baseline (base calculator features only)
  Parallel Jobs: 31 (using 32 CPUs)
  MC-CV Splits: 25
  Training Proportion: 80%

Training Process:
  1. Monte Carlo Cross-Validation (25 splits)
  2. Model Training: All three model types (CatBoost, XGBoost, XGBoost RF)
  3. Model Selection: Best model by C-index, then AU-PRC
  4. Final Model: Best model trained on full temporal split
--------------------------------------------------------------------------------

Model Features:
  - primary_etiology feature to distinguish etiologies
  - All derived variables (vad_combined, vent_combined, donor ratios, chd_lat)
  - All required input features from risk calculator
--------------------------------------------------------------------------------

Training Baseline Combined model (for all cohorts)...
  Output directory: outputs/models/Combined_base/


2026-02-03 16:37:12,438 - run_shap_ffa_workflow - INFO - FFA modules loaded using direct file import
2026-02-03 16:37:12,439 - train_python_models - INFO - Loading calculator data...
2026-02-03 16:37:12,439 - run_shap_ffa_workflow - INFO - Loading calculator data from /home/pgx3874/phts/graft-loss/data/phts_txpl_ml.sas7bdat
2026-02-03 16:37:12,653 - run_shap_ffa_workflow - INFO - Loaded 5835 rows for cohort Combined
2026-02-03 16:37:12,653 - run_shap_ffa_workflow - INFO - Preparing calculator features (eGFR, BMI, dichotomous variables, etc.)...
2026-02-03 16:37:12,684 - run_shap_ffa_workflow - INFO - Calculated egfr_tx using Schwartz formula
2026-02-03 16:37:12,686 - run_shap_ffa_workflow - INFO - Calculated egfr_listing using Schwartz formula
2026-02-03 16:37:12,688 - run_shap_ffa_workflow - INFO - Calculated bmi_txpl
2026-02-03 16:37:12,689 - run_shap_ffa_workflow - INFO - Calculated age_txpl_months from age_txpl
2026-02-03 16:37:12,692 - run_shap_ffa_workflow - INFO - Created egfr_t


✓ Baseline Combined model training complete!
  Total time: 0.0 minutes (1 seconds)
  Output saved to: outputs/models/Combined_base/

All three models have been trained:
  - CatBoost
  - XGBoost
  - XGBoost Random Forest

The best model (highest C-index) has been selected and saved.

The model is now ready for:
  - Risk prediction for all cohorts (CHD, Cardiomyopathy, Myocarditis)
  - SHAP/FFA analysis to extract causal factors

Baseline Model Training Complete!


In [9]:
# Check training results
import json

print("\nTraining Results Summary:")
print("-" * 80)

# Check both baseline and enhanced model directories
model_variants = [
    ("Baseline", f"{COHORT}_base"),
    ("Enhanced", f"{COHORT}_enhanced")
]

found_any = False
for variant_name, variant_dir in model_variants:
    best_model_file = CALCULATOR_DIR / "outputs" / "models" / variant_dir / "best_model.txt"
    if best_model_file.exists():
        found_any = True
        print(f"\n{variant_name} Model ({variant_dir}):")
        with open(best_model_file, 'r') as f:
            content = f.read()
            print(content)
            
            # Extract model performance info
            lines = content.split('\n')
            model_performances = {}
            for line in lines:
                if 'CatBoost C-index:' in line or 'CatBoost' in line and 'C-index' in line:
                    # Try to extract C-index value
                    parts = line.split(':')
                    if len(parts) > 1:
                        model_performances['CatBoost'] = parts[-1].strip().split()[0]
                elif 'XGBoost RF C-index:' in line or ('XGBoost RF' in line and 'C-index' in line):
                    parts = line.split(':')
                    if len(parts) > 1:
                        model_performances['XGBoost RF'] = parts[-1].strip().split()[0]
                elif 'XGBoost C-index:' in line or ('XGBoost' in line and 'C-index' in line and 'RF' not in line):
                    parts = line.split(':')
                    if len(parts) > 1:
                        model_performances['XGBoost'] = parts[-1].strip().split()[0]
            
            # Also try to extract from "Best Model (MC-CV):" line
            for line in lines:
                if 'Best Model (MC-CV):' in line:
                    best_model_name = line.split(':')[1].strip() if ':' in line else ''
                    # Try to find C-index for this model
                    for perf_line in lines:
                        if best_model_name in perf_line and 'C-index' in perf_line:
                            parts = perf_line.split(':')
                            if len(parts) > 1:
                                c_index_val = parts[-1].strip().split()[0]
                                if best_model_name not in model_performances:
                                    model_performances[best_model_name] = c_index_val
        
        if model_performances:
            print(f"\n  Model Performance Comparison:")
            for model_name, c_index in sorted(model_performances.items(), 
                                               key=lambda x: float(x[1]) if x[1].replace('.', '').isdigit() else 0, 
                                               reverse=True):
                print(f"    {model_name:20s} C-index: {c_index}")
        
        # List model files for this variant
        models_dir = CALCULATOR_DIR / "outputs" / "models" / variant_dir
        if models_dir.exists():
            catboost_file = models_dir / "catboost_model.cbm"
            xgboost_file = models_dir / "xgboost_model.ubj"
            xgboost_rf_file = models_dir / "xgboost_rf_model.ubj"
            
            print(f"\n  Trained Models ({variant_name}):")
            if catboost_file.exists():
                size_mb = catboost_file.stat().st_size / (1024 * 1024)
                print(f"    ✓ CatBoost: {catboost_file.name} ({size_mb:.2f} MB)")
            else:
                print(f"    ○ CatBoost: Not found")
            
            if xgboost_file.exists():
                size_mb = xgboost_file.stat().st_size / (1024 * 1024)
                print(f"    ✓ XGBoost: {xgboost_file.name} ({size_mb:.2f} MB)")
            else:
                print(f"    ○ XGBoost: Not found")
            
            if xgboost_rf_file.exists():
                size_mb = xgboost_rf_file.stat().st_size / (1024 * 1024)
                print(f"    ✓ XGBoost RF: {xgboost_rf_file.name} ({size_mb:.2f} MB)")
            else:
                print(f"    ○ XGBoost RF: Not found")
            
            # Check feature count
            feature_file = models_dir / "feature_names.json"
            if feature_file.exists():
                with open(feature_file, 'r') as f:
                    features = json.load(f)
                    print(f"\n  Feature Summary ({variant_name}):")
                    print(f"    Total features: {len(features)}")
                    print(f"    ✓ primary_etiology: {'primary_etiology' in features or any('primary_etiology' in f for f in features)}")
                    print(f"    ✓ vad_combined: {'vad_combined' in features}")
                    print(f"    ✓ vent_combined: {'vent_combined' in features}")
                    print(f"    ✓ ecmo_combined: {'ecmo_combined' in features}")
                    print(f"    ✓ donor_weight_ratio: {'donor_weight_ratio' in features}")
                    print(f"    ✓ donor_size_ratio: {'donor_size_ratio' in features}")
                    print(f"    ✓ chd_lat: {'chd_lat' in features}")

if not found_any:
    print(f"\n⚠ {COHORT}: Best model files not found")
    print(f"  Checked directories:")
    for variant_name, variant_dir in model_variants:
        checked_path = CALCULATOR_DIR / "outputs" / "models" / variant_dir / "best_model.txt"
        print(f"    - {variant_dir}/best_model.txt: {'✓ Found' if checked_path.exists() else '✗ Not found'}")


Training Results Summary:
--------------------------------------------------------------------------------

Baseline Model (Combined_base):
Best Model (MC-CV): XGBoost
Selection Criteria: C-index (primary), AU-PRC (tiebreaker)
MC-CV Mean C-index: 0.635117
MC-CV 95% CI: [0.604652, 0.662200]
MC-CV SD: 0.015089
MC-CV Mean AUC: 0.608229 ± 0.016370
MC-CV Mean AU-PRC: 0.235276 ± 0.016221
MC-CV Mean Recall: 0.640851 ± 0.089856
MC-CV n_splits: 25

Temporal Split Results:
  CatBoost: 0.620362
  XGBoost: 0.688172
  XGBoost RF: 0.641651

MC-CV Model Performance (all models):
  CatBoost:
    C-index: 0.568481 ± 0.061285 (95% CI: 0.440812 - 0.652181)
    AUC: 0.558459 ± 0.043782 (95% CI: 0.476635 - 0.629679)
    AU-PRC: 0.206219 ± 0.027650 (95% CI: 0.156437 - 0.253772)
    Recall: 0.537872 ± 0.184974 (95% CI: 0.177660 - 0.858511)
    [25 splits]
  XGBoost:
    C-index: 0.635117 ± 0.015089 (95% CI: 0.604652 - 0.662200)
    AUC: 0.608229 ± 0.016370 (95% CI: 0.577097 - 0.636627)
    AU-PRC: 0.235276 

### b. Enhanced Model (with Recommended Features)

Train the enhanced calculator model with recommended additional features (BNP, CRP, sec_dx/ter_dx, lipid panel, etc.) to compare performance with the base model.

**Enhanced Features Include:**
- BNP (Brain Natriuretic Peptide): `txbnp`, `txpbnp_r`, `lbnp`, `lspbnp_r`
- CRP (C-Reactive Protein): `txcrp_r`, `lcrp_r`
- Secondary/Tertiary diagnoses: `sec_dx`, `ter_dx`
- Pre-albumin at listing: `lspalb_r`
- Lipid panel: `txchol_r`, `txtg_r`, `txldl_r`, `txhdl_r`, `txvldl_r`
- Oxygen saturation: `txbaosat`, `txsvcsat`, `lsbaosat`, `lssvcsat`

**Note:** This model will be saved to `outputs/models/Combined_enhanced/` for comparison with the base model.

In [10]:
# Import training function
from train_python_models import train_models_for_cohort
import os
import multiprocessing

# Determine number of parallel jobs (use all available CPUs minus 1 for safety)
n_parallel_jobs = max(1, multiprocessing.cpu_count() - 1)

print("=" * 80)
print("Training Enhanced Calculator Model (with Recommended Features)")
print("=" * 80)
print(f"\nConfiguration:")
print(f"  Cohort: {COHORT}")
print(f"  Feature Set: Enhanced (base + recommended features)")
print(f"  Parallel Jobs: {n_parallel_jobs} (using {multiprocessing.cpu_count()} CPUs)")
print(f"  MC-CV Splits: 25")
print(f"  Training Proportion: 80%")
print(f"\nEnhanced Features:")
print(f"  - BNP values (txbnp, txpbnp_r, lbnp, lspbnp_r)")
print(f"  - CRP (txcrp_r, lcrp_r)")
print(f"  - Secondary/Tertiary diagnoses (sec_dx, ter_dx)")
print(f"  - Pre-albumin at listing (lspalb_r)")
print(f"  - Lipid panel (txchol_r, txtg_r, txldl_r, txhdl_r, txvldl_r)")
print(f"  - Oxygen saturation (txbaosat, txsvcsat, lsbaosat, lssvcsat)")

print(f"\nTraining Process:")
print(f"  1. Monte Carlo Cross-Validation (25 splits)")
print(f"  2. Model Training: All three model types (CatBoost, XGBoost, XGBoost RF)")
print(f"  3. Model Selection: Best model by C-index, then AU-PRC")
print(f"  4. Final Model: Best model trained on full temporal split")

print(f"\nTraining Enhanced Combined model (for all cohorts)...")
print(f"  Output directory: outputs/models/Combined_enhanced/")

try:
    import time
    start_time = time.time()
    
    train_models_for_cohort(
        cohort=COHORT,
        n_mc_splits=25,
        train_prop=0.8,
        n_jobs=n_parallel_jobs,  # Use parallel processing
        include_recommended_features=True  # Enable enhanced features
    )
    
    elapsed_time = time.time() - start_time
    print(f"\n✓ Enhanced Combined model training complete!")
    print(f"  Total time: {elapsed_time/60:.1f} minutes ({elapsed_time:.0f} seconds)")
    print(f"  Output saved to: outputs/models/Combined_enhanced/")
    
except Exception as e:
    print(f"\n✗ Error training Enhanced Combined model: {e}")
    import traceback
    traceback.print_exc()

print("\n" + "=" * 80)
print("Enhanced Model Training Complete!")
print("=" * 80)

2026-02-03 16:37:41,394 - train_python_models - INFO - 
2026-02-03 16:37:41,395 - train_python_models - INFO - Training models for cohort: Combined
2026-02-03 16:37:41,395 - train_python_models - INFO - ================================================================================
2026-02-03 16:37:41,395 - train_python_models - INFO - MC-CV Configuration:
2026-02-03 16:37:41,396 - train_python_models - INFO -   - Number of splits: 25
2026-02-03 16:37:41,396 - train_python_models - INFO -   - Training proportion: 80.0%
2026-02-03 16:37:41,396 - train_python_models - INFO -   - Parallel jobs: 31
2026-02-03 16:37:41,396 - train_python_models - INFO -   - Time horizon for AUC/AU-PRC/Recall: 365.25 days
2026-02-03 16:37:41,397 - train_python_models - INFO - 
2026-02-03 16:37:41,397 - train_python_models - INFO - Loading calculator data...
2026-02-03 16:37:41,397 - run_shap_ffa_workflow - INFO - Loading calculator data from /home/pgx3874/phts/graft-loss/data/phts_txpl_ml.sas7bdat
2026-02-0

Training Enhanced Calculator Model (with Recommended Features)

Configuration:
  Cohort: Combined
  Feature Set: Enhanced (base + recommended features)
  Parallel Jobs: 31 (using 32 CPUs)
  MC-CV Splits: 25
  Training Proportion: 80%

Enhanced Features:
  - BNP values (txbnp, txpbnp_r, lbnp, lspbnp_r)
  - CRP (txcrp_r, lcrp_r)
  - Secondary/Tertiary diagnoses (sec_dx, ter_dx)
  - Pre-albumin at listing (lspalb_r)
  - Lipid panel (txchol_r, txtg_r, txldl_r, txhdl_r, txvldl_r)
  - Oxygen saturation (txbaosat, txsvcsat, lsbaosat, lssvcsat)

Training Process:
  1. Monte Carlo Cross-Validation (25 splits)
  2. Model Training: All three model types (CatBoost, XGBoost, XGBoost RF)
  3. Model Selection: Best model by C-index, then AU-PRC
  4. Final Model: Best model trained on full temporal split

Training Enhanced Combined model (for all cohorts)...
  Output directory: outputs/models/Combined_enhanced/


2026-02-03 16:37:41,614 - run_shap_ffa_workflow - INFO - Calculated egfr_tx using Schwartz formula
2026-02-03 16:37:41,615 - run_shap_ffa_workflow - INFO - Calculated egfr_listing using Schwartz formula
2026-02-03 16:37:41,617 - run_shap_ffa_workflow - INFO - Calculated bmi_txpl
2026-02-03 16:37:41,619 - run_shap_ffa_workflow - INFO - Created egfr_tx_cat categories
2026-02-03 16:37:41,620 - run_shap_ffa_workflow - INFO - Created egfr_listing_cat categories
2026-02-03 16:37:41,621 - run_shap_ffa_workflow - INFO - Created txbili_t_r_high
2026-02-03 16:37:41,621 - run_shap_ffa_workflow - INFO - Created txbun_r_high from txbun_r
2026-02-03 16:37:41,622 - run_shap_ffa_workflow - INFO - Created txsa_r_low
2026-02-03 16:37:41,622 - run_shap_ffa_workflow - INFO - Created txalt_high
2026-02-03 16:37:41,623 - run_shap_ffa_workflow - INFO - Created ecmo_combined
2026-02-03 16:37:41,624 - run_shap_ffa_workflow - INFO - Created vad_combined
2026-02-03 16:37:41,625 - run_shap_ffa_workflow - INFO - C


✓ Enhanced Combined model training complete!
  Total time: 0.0 minutes (0 seconds)
  Output saved to: outputs/models/Combined_enhanced/

Enhanced Model Training Complete!


## 4. Causal Analysis Workflow

Generate SHAP values and extract causal factors using Formal Feature Attribution for both Baseline and Enhanced models.

**Note:** This section has two parts:
- **4a. Baseline Model SHAP/FFA** - Analyzes `Combined_base` model (outputs to `outputs/shap_ffa/Combined_base/`)
- **4b. Enhanced Model SHAP/FFA** - Analyzes `Combined_enhanced` model (outputs to `outputs/shap_ffa/Combined_enhanced/`)

Both analyses are required for the dashboard to display results for both model tabs.

4a. Baseline Model SHAP/FFA Analysis

Generate SHAP values and extract causal factors for the **Baseline Model** (`Combined_base`).

**Step 1: SHAP Value Computation**

The workflow checks which model is best (from `best_model.txt` in `Combined_base/`):

- **If XGBoost is best model:**
  - ✅ Computes SHAP values from **best XGBoost model** only
  - Uses simplified pipeline (XGBoost SHAP only)
  - No weights needed (single model)

- **If CatBoost is best model:**
  - ✅ Computes SHAP values from **best CatBoost model**
  - ✅ Computes SHAP values from **best XGBoost model**
  - ✅ **Automatically determines weights** based on C-index values:
    - Weights are calculated from relative C-index performance
    - CatBoost (best model) gets higher weight
    - XGBoost gets lower weight
    - Weights normalized to sum to 1.0
  - Uses combined pipeline (CatBoost + XGBoost SHAP)

**Step 2: FFA Rule Extraction**

- ✅ **Always uses**: Best XGBoost JSON model for rule extraction
  - Rules are extracted from XGBoost JSON structure (`*_final_model_xgboost.json`)
  - CatBoost JSON is **never** used for rule extraction (harder to parse due to categorical hashing)
  - Even if CatBoost is the best model, rules come from XGBoost JSON

**Step 3: Rule Filtering & Causal Responsibility**

- Rules are filtered using SHAP importance values (from Step 1)
- Causal responsibility is calculated as:
  ```
  causal_responsibility = (rule_frequency / total_rules) × SHAP_importance
  ```
- Where `SHAP_importance` comes from:
  - XGBoost SHAP only (if XGBoost is best), OR
  - Combined CatBoost + XGBoost SHAP (if CatBoost is best)

**Note:** FFA analysis is **REQUIRED**. The workflow requires `ffa_analysis/` directory with:
- `ffa_utils.py` (with `load_model_json`, `extract_feature_mappings`)
- `xgboost_axp_explainer.py` (with `XGBoostSymbolicExplainer`, `PathConfig`)

These modules have been restored from git history and are now available in the repository.

**Summary - Explicit Model Usage:**

| Component | Model Used | Condition |
|-----------|-----------|-----------|
| **SHAP Values** | Best XGBoost | Always |
| **SHAP Values** | Best CatBoost | Only if CatBoost is best model |
| **Rule Extraction** | Best XGBoost JSON | Always (regardless of which model is best) |
| **FFA Analysis** | Best XGBoost JSON + SHAP | Always |

**Key Point:** Even if CatBoost is the best model, the FFA analysis still uses the **XGBoost JSON model** for rule extraction, but filters those rules using **combined SHAP values** (CatBoost + XGBoost).

### 4a. Baseline Model Shap/FFA Analysis

In [11]:
# Run SHAP/FFA workflow for Baseline Combined model
import subprocess

print(f"\n{'=' * 80}")
print("Running SHAP + FFA Analysis for Baseline Model")
print(f"{'=' * 80}")
print(f"Analyzing Baseline Combined model (Combined_base)...")
print("-" * 80)
print(f"\nCausal Analysis Process:")
print(f"  1. Check best model (from best_model.txt in Combined_base/)")
print(f"  2. Compute SHAP values:")
print(f"     - If XGBoost is best: XGBoost SHAP only")
print(f"     - If CatBoost is best: Combined SHAP (CatBoost + XGBoost)")
print(f"     - Weights auto-determined from C-index values")
print(f"  3. Extract rules from best XGBoost JSON model (if FFA available)")
print(f"  4. Filter rules using SHAP importance")
print(f"  5. Calculate causal responsibility (if FFA available)")
print(f"  6. Generate top {TOP_K} causal factors")
print("-" * 80)
print(f"\nModel Variant: Baseline (base calculator features only)")
print(f"Output: outputs/shap_ffa/Combined_base/ (dashboard data for Baseline Model tab)")
print("-" * 80)
print(f"\nNote: If FFA modules are not available:")
print(f"  - Workflow continues with SHAP analysis only")
print(f"  - Uses SHAP importance instead of causal responsibility")
print(f"  - Dashboard will show SHAP-based rankings")
print("-" * 80)

# Build command - explicitly specify baseline model variant
cmd = [
    sys.executable,
    str(CALCULATOR_DIR / "run_shap_ffa_workflow.py"),
    "--cohort", COHORT,
    "--model-variant", "base",  # Explicitly use baseline model
    "--top-k", str(TOP_K)
]

# Only add weight arguments if manually specified (otherwise auto-determined)
if WEIGHT_CATBOOST is not None:
    cmd.extend(["--weight-catboost", str(WEIGHT_CATBOOST)])
if WEIGHT_XGBOOST is not None:
    cmd.extend(["--weight-xgboost", str(WEIGHT_XGBOOST)])

try:
    result = subprocess.run(
        cmd,
        cwd=str(CALCULATOR_DIR),
        capture_output=False,  # Show output in real-time
        text=True
    )
    
    if result.returncode == 0:
        print(f"\n✓ Baseline model SHAP/FFA analysis complete!")
        print(f"\nResults include:")
        print(f"  - Top {TOP_K} causal factors with causal responsibility scores")
        print(f"  - Feature importance rankings (SHAP-based)")
        print(f"  - Rule-based FFA analysis results")
        print(f"  - Dashboard data for Baseline Model tab (outputs/shap_ffa/Combined_base/)")
        print(f"\nNote: Causal factors are calculated using:")
        print(f"  - Rules extracted from best XGBoost JSON model (Combined_base/)")
        print(f"  - SHAP importance from best models (XGBoost, and CatBoost if CatBoost is best)")
        print(f"  - Formula: causal_responsibility = (rule_frequency / total_rules) × SHAP_importance")
    else:
        print(f"\n⚠ SHAP/FFA exited with code: {result.returncode}")
except Exception as e:
    print(f"\n✗ Error running SHAP/FFA: {e}")
    logger.error(f"Error running SHAP/FFA", exc_info=True)

print(f"\n{'=' * 80}")
print("Baseline Model SHAP/FFA analysis complete!")
print(f"{'=' * 80}")


Running SHAP + FFA Analysis for Baseline Model
Analyzing Baseline Combined model (Combined_base)...
--------------------------------------------------------------------------------

Causal Analysis Process:
  1. Check best model (from best_model.txt in Combined_base/)
  2. Compute SHAP values:
     - If XGBoost is best: XGBoost SHAP only
     - If CatBoost is best: Combined SHAP (CatBoost + XGBoost)
     - Weights auto-determined from C-index values
  3. Extract rules from best XGBoost JSON model (if FFA available)
  4. Filter rules using SHAP importance
  5. Calculate causal responsibility (if FFA available)
  6. Generate top 10 causal factors
--------------------------------------------------------------------------------

Model Variant: Baseline (base calculator features only)
Output: outputs/shap_ffa/Combined_base/ (dashboard data for Baseline Model tab)
--------------------------------------------------------------------------------

Note: If FFA modules are not available:
  - Wo

2026-02-03 16:37:56,395 - __main__ - INFO - FFA modules loaded using direct file import
2026-02-03 16:37:56,630 - botocore.credentials - INFO - Found credentials from IAM Role: EC2_Spot
2026-02-03 16:37:56,670 - __main__ - INFO - Using model variant: base -> model directory: Combined_base
2026-02-03 16:37:56,670 - __main__ - INFO - Output directory: /home/pgx3874/phts/graft-loss/cohort_analysis/calculator/outputs/shap_ffa/Combined_base
2026-02-03 16:37:56,670 - __main__ - WARNING - Unknown best model 'None', using default weights: CatBoost=0.6, XGBoost=0.4
2026-02-03 16:37:56,670 - __main__ - INFO - Auto-determined weights from best model: CatBoost=0.600, XGBoost=0.400
2026-02-03 16:37:56,670 - __main__ - INFO - ================================================================================
2026-02-03 16:37:56,670 - __main__ - INFO - SHAP + FFA Analysis for Combined Cohort
2026-02-03 16:37:56,670 - __main__ - INFO - =====================================================================


⚠ SHAP/FFA exited with code: 1

Baseline Model SHAP/FFA analysis complete!


### 4b. Extended Model Shap/FFA Analysis

In [ ]:
# Run SHAP/FFA workflow for Extended model
import subprocess

print(f"\n{'=' * 80}")
print("Running SHAP + FFA Analysis for Extended Model")
print(f"{'=' * 80}")
print(f"Analyzing Extended Combined model (for all cohorts)...")
print("-" * 80)
print(f"\nCausal Analysis Process:")
print(f"  1. Check best model (from best_model.txt in Combined_enhanced/)")
print(f"  2. Compute SHAP values:")
print(f"     - If XGBoost is best: XGBoost SHAP only")
print(f"     - If CatBoost is best: Combined SHAP (CatBoost + XGBoost)")
print(f"     - Weights auto-determined from C-index values")
print(f"  3. Extract rules from best XGBoost JSON model (if FFA available)")
print(f"  4. Filter rules using SHAP importance")
print(f"  5. Calculate causal responsibility (if FFA available)")
print(f"  6. Generate top {TOP_K} causal factors")
print("-" * 80)
print(f"\nModel Variant: Enhanced (base + recommended features)")
print(f"Output: outputs/shap_ffa/Combined_enhanced/ (dashboard data for Extended Model tab)")
print("-" * 80)
print(f"\nNote: The workflow will use models from Combined_enhanced/ directory.")
print(f"      Make sure Enhanced model training (Step 3b) completed successfully.")
print("-" * 80)

# Build command - explicitly specify enhanced model variant
cmd = [
    sys.executable,
    str(CALCULATOR_DIR / "run_shap_ffa_workflow.py"),
    "--cohort", COHORT,
    "--model-variant", "enhanced",  # Explicitly use enhanced model
    "--top-k", str(TOP_K)
]

# Only add weight arguments if manually specified (otherwise auto-determined)
if WEIGHT_CATBOOST is not None:
    cmd.extend(["--weight-catboost", str(WEIGHT_CATBOOST)])
if WEIGHT_XGBOOST is not None:
    cmd.extend(["--weight-xgboost", str(WEIGHT_XGBOOST)])

try:
    result = subprocess.run(
        cmd,
        cwd=str(CALCULATOR_DIR),
        capture_output=False,  # Show output in real-time
        text=True
    )
    
    if result.returncode == 0:
        print(f"\n✓ Extended model SHAP/FFA analysis complete!")
        print(f"\nResults include:")
        print(f"  - Top {TOP_K} causal factors with causal responsibility scores")
        print(f"  - Feature importance rankings (SHAP-based)")
        print(f"  - Rule-based FFA analysis results")
        print(f"  - Dashboard data for Extended Model tab")
        print(f"\nNote: Causal factors are calculated using:")
        print(f"  - Rules extracted from best XGBoost JSON model (Combined_enhanced/)")
        print(f"  - SHAP importance from best models (XGBoost, and CatBoost if CatBoost is best)")
        print(f"  - Formula: causal_responsibility = (rule_frequency / total_rules) × SHAP_importance")
    else:
        print(f"\n⚠ SHAP/FFA exited with code: {result.returncode}")
except Exception as e:
    print(f"\n✗ Error running SHAP/FFA: {e}")
    logger.error(f"Error running SHAP/FFA", exc_info=True)

print(f"\n{'=' * 80}")
print("Extended Model SHAP/FFA analysis complete!")
print(f"{'=' * 80}")

## 5. Inspect Results

View top causal factors, feature importance, and dashboard data for the Combined model.

In [ ]:
# Load and display dashboard data for both Baseline and Enhanced models
import json
import pandas as pd

print("\n" + "=" * 80)
print("Results Summary - Baseline and Enhanced Models")
print("=" * 80)

# Check both model variants
model_variants = [
    ("Baseline", f"{COHORT}_base"),
    ("Enhanced", f"{COHORT}_enhanced")
]

for variant_name, variant_dir in model_variants:
    print(f"\n{'=' * 80}")
    print(f"{variant_name} Model ({variant_dir}) - Top {TOP_K} Causal Factors")
    print("=" * 80)
    
    dashboard_data_file = (
        CALCULATOR_DIR / "outputs" / "shap_ffa" / variant_dir / "dashboard_data.json"
    )
    
    if dashboard_data_file.exists():
        with open(dashboard_data_file, 'r') as f:
            dashboard_data = json.load(f)
        
        top_factors = dashboard_data.get('top_causal_factors', [])[:TOP_K]
        
        if top_factors:
            for idx, factor in enumerate(top_factors, 1):
                importance = factor.get('causal_responsibility', 
                                     factor.get('importance', 
                                               factor.get('combined_importance_norm', 0)))
                print(f"{idx:2d}. {factor['feature']:40s} "
                      f"(Importance: {importance:.4f})")
        else:
            print("  (No causal factors available)")
        
        # Display summary statistics
        if 'summary' in dashboard_data:
            print(f"\n  Summary Statistics:")
            summary = dashboard_data['summary']
            for key, value in summary.items():
                print(f"    {key}: {value}")
        
        # Check for key features in top factors
        print(f"\n  Key Features Check:")
        top_feature_names = [f['feature'] for f in top_factors]
        key_features = {
            'primary_etiology': any('primary_etiology' in f for f in top_feature_names),
            'vad_combined': 'vad_combined' in top_feature_names,
            'vent_combined': 'vent_combined' in top_feature_names,
            'ecmo_combined': 'ecmo_combined' in top_feature_names,
            'donor_weight_ratio': 'donor_weight_ratio' in top_feature_names,
            'donor_size_ratio': 'donor_size_ratio' in top_feature_names,
            'chd_lat': 'chd_lat' in top_feature_names,
            'egfr_tx': 'egfr_tx' in top_feature_names or any('egfr' in f for f in top_feature_names),
            'txfcpra': 'txfcpra' in top_feature_names,
            'hxsurg': 'hxsurg' in top_feature_names
        }
        for feature, present in key_features.items():
            status = "✓" if present else "○"
            print(f"    {status} {feature}")
    else:
        print(f"\n⚠ Dashboard data not found for {variant_name} model")
        print(f"  Expected: {dashboard_data_file}")
        if variant_name == "Baseline":
            print("  Run SHAP/FFA analysis for baseline model first (Section 4a)")
        else:
            print("  Run SHAP/FFA analysis for enhanced model first (Section 4b)")

### a. Feature Importances

In [ ]:
# Load and display feature importance for both Baseline and Enhanced models
print("\n" + "=" * 80)
print("Feature Importance Rankings - Baseline and Enhanced Models")
print("=" * 80)

# Check both model variants
model_variants = [
    ("Baseline", f"{COHORT}_base"),
    ("Enhanced", f"{COHORT}_enhanced")
]

for variant_name, variant_dir in model_variants:
    print(f"\n{'=' * 80}")
    print(f"{variant_name} Model ({variant_dir}) - Feature Importance")
    print("=" * 80)
    
    importance_files = list(
        (CALCULATOR_DIR / "outputs" / "models" / variant_dir).glob("importance_*.csv")
    )
    
    if importance_files:
        for imp_file in sorted(importance_files):
            model_name = imp_file.stem.replace(f"importance_{variant_dir}_", "")
            print(f"\n  {model_name}:")
            df = pd.read_csv(imp_file)
            print(f"    Total features: {len(df)}")
            print(f"    Top 10 features:")
            top10 = df.nlargest(10, 'importance')
            for idx, row in top10.iterrows():
                print(f"      {row['feature']:40s} {row['importance']:.4f}")
            
            # Check for key features
            feature_list = df['feature'].tolist()
            print(f"\n    Key Features Status:")
            key_features = {
                'primary_etiology': any('primary_etiology' in f for f in feature_list),
                'vad_combined': 'vad_combined' in feature_list,
                'vent_combined': 'vent_combined' in feature_list,
                'ecmo_combined': 'ecmo_combined' in feature_list,
                'donor_weight_ratio': 'donor_weight_ratio' in feature_list,
                'donor_size_ratio': 'donor_size_ratio' in feature_list,
                'chd_lat': 'chd_lat' in feature_list,
                'egfr_tx': 'egfr_tx' in feature_list,
                'txfcpra': 'txfcpra' in feature_list,
                'hxsurg': 'hxsurg' in feature_list
            }
            for feature, present in key_features.items():
                status = "✓" if present else "○"
                if present:
                    rank = df[df['feature'] == feature].index[0] + 1 if feature in feature_list else "N/A"
                    print(f"      {status} {feature:25s} (Rank: {rank})")
                else:
                    print(f"      {status} {feature:25s} (Not found)")
    else:
        print(f"\n⚠ No feature importance files found for {variant_name} model")
        if variant_name == "Baseline":
            print(f"  Train Baseline model first (Section 3a)")
        else:
            print(f"  Train Enhanced model first (Section 3b)")

### b. Visualizations

Create visualizations comparing Baseline and Enhanced models for causal importance.

In [ ]:
# Plot top causal factors comparison: Baseline vs Enhanced models
try:
    import matplotlib.pyplot as plt
    import numpy as np
    
    # Load dashboard data for both model variants
    base_dashboard_file = (
        CALCULATOR_DIR / "outputs" / "shap_ffa" / f"{COHORT}_base" / "dashboard_data.json"
    )
    enhanced_dashboard_file = (
        CALCULATOR_DIR / "outputs" / "shap_ffa" / f"{COHORT}_enhanced" / "dashboard_data.json"
    )
    
    base_data = None
    enhanced_data = None
    
    if base_dashboard_file.exists():
        with open(base_dashboard_file, 'r') as f:
            base_data = json.load(f)
        print(f"\n✓ Loaded Baseline model data from: {base_dashboard_file}")
    else:
        print(f"\n⚠ Baseline dashboard data not found: {base_dashboard_file}")
        print("  Run SHAP/FFA analysis for baseline model first (Section 3)")
    
    if enhanced_dashboard_file.exists():
        with open(enhanced_dashboard_file, 'r') as f:
            enhanced_data = json.load(f)
        print(f"✓ Loaded Enhanced model data from: {enhanced_dashboard_file}")
    else:
        print(f"⚠ Enhanced dashboard data not found: {enhanced_dashboard_file}")
        print("  Run SHAP/FFA analysis for enhanced model first (Section 4)")
    
    # Create comparison visualizations if both models are available
    if base_data and enhanced_data:
        base_factors = base_data.get('top_causal_factors', [])[:TOP_K]
        enhanced_factors = enhanced_data.get('top_causal_factors', [])[:TOP_K]
        
        if base_factors and enhanced_factors:
            # Get all unique features from both models
            all_features = set()
            base_factors_dict = {}
            enhanced_factors_dict = {}
            
            for f in base_factors:
                feature = f['feature']
                all_features.add(feature)
                base_factors_dict[feature] = f.get('causal_responsibility', 
                                                   f.get('importance', 
                                                        f.get('combined_importance_norm', 0)))
            
            for f in enhanced_factors:
                feature = f['feature']
                all_features.add(feature)
                enhanced_factors_dict[feature] = f.get('causal_responsibility', 
                                                      f.get('importance', 
                                                           f.get('combined_importance_norm', 0)))
            
            # Sort by combined importance (average of both models)
            feature_list = sorted(all_features, 
                                key=lambda x: (base_factors_dict.get(x, 0) + enhanced_factors_dict.get(x, 0)) / 2,
                                reverse=True)[:TOP_K]
            
            base_importance = [base_factors_dict.get(f, 0) for f in feature_list]
            enhanced_importance = [enhanced_factors_dict.get(f, 0) for f in feature_list]
            
            # Create comparison plot
            fig, axes = plt.subplots(1, 2, figsize=(16, max(8, len(feature_list) * 0.5)))
            
            # Left plot: Side-by-side bars
            ax1 = axes[0]
            x = np.arange(len(feature_list))
            width = 0.35
            
            bars1 = ax1.barh(x - width/2, base_importance, width, 
                            label='Baseline Model', color='#3b82f6', alpha=0.8)
            bars2 = ax1.barh(x + width/2, enhanced_importance, width, 
                            label='Enhanced Model', color='#10b981', alpha=0.8)
            
            ax1.set_yticks(x)
            ax1.set_yticklabels(feature_list)
            ax1.set_xlabel('Causal Responsibility / Importance')
            ax1.set_title(f'Top {TOP_K} Causal Factors Comparison')
            ax1.legend()
            ax1.invert_yaxis()
            ax1.grid(axis='x', alpha=0.3)
            
            # Right plot: Difference (Enhanced - Baseline)
            ax2 = axes[1]
            differences = [enhanced_importance[i] - base_importance[i] 
                          for i in range(len(feature_list))]
            colors = ['#10b981' if d > 0 else '#ef4444' if d < 0 else '#64748b' 
                     for d in differences]
            
            bars3 = ax2.barh(range(len(feature_list)), differences, color=colors, alpha=0.8)
            ax2.set_yticks(range(len(feature_list)))
            ax2.set_yticklabels(feature_list)
            ax2.set_xlabel('Difference (Enhanced - Baseline)')
            ax2.set_title('Importance Difference')
            ax2.axvline(x=0, color='black', linestyle='--', linewidth=1)
            ax2.invert_yaxis()
            ax2.grid(axis='x', alpha=0.3)
            
            plt.tight_layout()
            
            # Save plot
            plot_file = CALCULATOR_DIR / "outputs" / "shap_ffa" / f"{COHORT}_comparison_top_{TOP_K}_factors.png"
            plt.savefig(plot_file, dpi=150, bbox_inches='tight')
            print(f"\n✓ Saved comparison plot: {plot_file}")
            
            plt.show()
            
            # Print summary statistics
            print(f"\n{'='*80}")
            print(f"Comparison Summary - Top {TOP_K} Causal Factors")
            print(f"{'='*80}")
            print(f"\n{'Feature':<40} {'Baseline':<12} {'Enhanced':<12} {'Difference':<12}")
            print("-" * 80)
            for i, feature in enumerate(feature_list):
                base_val = base_importance[i]
                enh_val = enhanced_importance[i]
                diff = differences[i]
                print(f"{feature:<40} {base_val:>10.4f}   {enh_val:>10.4f}   {diff:>+10.4f}")
            
        else:
            print("\n⚠ No causal factors available for comparison")
    elif base_data:
        # Only baseline available - plot single model
        top_factors = base_data.get('top_causal_factors', [])[:TOP_K]
        if top_factors:
            features = [f['feature'] for f in top_factors]
            importance = [f.get('causal_responsibility', 
                              f.get('importance', 
                                   f.get('combined_importance_norm', 0))) 
                        for f in top_factors]
            
            plt.figure(figsize=(10, max(6, len(features) * 0.4)))
            plt.barh(range(len(features)), importance, color='#3b82f6', alpha=0.8)
            plt.yticks(range(len(features)), features)
            plt.xlabel('Causal Responsibility / Importance')
            plt.title(f'Top {TOP_K} Causal Factors - Baseline Model')
            plt.gca().invert_yaxis()
            plt.tight_layout()
            
            plot_file = CALCULATOR_DIR / "outputs" / "shap_ffa" / f"{COHORT}_base" / f"top_{TOP_K}_factors.png"
            plt.savefig(plot_file, dpi=150, bbox_inches='tight')
            print(f"\n✓ Saved plot: {plot_file}")
            plt.show()
    elif enhanced_data:
        # Only enhanced available - plot single model
        top_factors = enhanced_data.get('top_causal_factors', [])[:TOP_K]
        if top_factors:
            features = [f['feature'] for f in top_factors]
            importance = [f.get('causal_responsibility', 
                              f.get('importance', 
                                   f.get('combined_importance_norm', 0))) 
                        for f in top_factors]
            
            plt.figure(figsize=(10, max(6, len(features) * 0.4)))
            plt.barh(range(len(features)), importance, color='#10b981', alpha=0.8)
            plt.yticks(range(len(features)), features)
            plt.xlabel('Causal Responsibility / Importance')
            plt.title(f'Top {TOP_K} Causal Factors - Enhanced Model')
            plt.gca().invert_yaxis()
            plt.tight_layout()
            
            plot_file = CALCULATOR_DIR / "outputs" / "shap_ffa" / f"{COHORT}_enhanced" / f"top_{TOP_K}_factors.png"
            plt.savefig(plot_file, dpi=150, bbox_inches='tight')
            print(f"\n✓ Saved plot: {plot_file}")
            plt.show()
    else:
        print("\n⚠ No dashboard data available for visualization")
        print("  Run SHAP/FFA analysis first (Sections 3 and 4)")
            
except ImportError:
    print("\n⚠ Matplotlib not available. Skipping visualizations.")
    print("  Install with: pip install matplotlib")
except Exception as e:
    print(f"\n⚠ Error creating visualizations: {e}")
    import traceback
    traceback.print_exc()

### c. Export Summary

Create a summary JSON file with all results for both Baseline and Enhanced models.

In [ ]:
# Create workflow summary
from datetime import datetime

summary = {
    "workflow": "Calculator Model Training + SHAP/FFA Analysis",
    "model_strategy": "Dual Combined models (Baseline + Enhanced) for all cohorts",
    "timestamp": datetime.now().isoformat(),
    "configuration": {
        "cohort": COHORT,
        "top_k": TOP_K,
        "weight_catboost": WEIGHT_CATBOOST,
        "weight_xgboost": WEIGHT_XGBOOST,
        "debug_mode": DEBUG_MODE
    },
    "models": {
        "baseline": {},
        "enhanced": {}
    }
}

# Process both model variants
model_variants = [
    ("baseline", f"{COHORT}_base"),
    ("enhanced", f"{COHORT}_enhanced")
]

for variant_name, variant_dir in model_variants:
    variant_summary = {}
    
    # Best model
    best_model_file = CALCULATOR_DIR / "outputs" / "models" / variant_dir / "best_model.txt"
if best_model_file.exists():
    with open(best_model_file, 'r') as f:
        content = f.read()
        lines = content.split('\n')
        for line in lines:
            if line.startswith("Best Model:"):
                variant_summary["best_model"] = line.replace("Best Model: ", "").strip()
            elif line.startswith("C-index:"):
                try:
                    variant_summary["c_index"] = float(line.replace("C-index: ", "").strip())
                except:
                    pass
    
    # Dashboard data
    dashboard_file = CALCULATOR_DIR / "outputs" / "shap_ffa" / variant_dir / "dashboard_data.json"
if dashboard_file.exists():
    with open(dashboard_file, 'r') as f:
        dashboard_data = json.load(f)
        variant_summary["top_factors_count"] = len(dashboard_data.get('top_causal_factors', []))
        if dashboard_data.get('top_causal_factors'):
            variant_summary["top_factor"] = dashboard_data['top_causal_factors'][0]['feature']
            variant_summary["top_factor_importance"] = dashboard_data['top_causal_factors'][0].get(
                'causal_responsibility', 
                dashboard_data['top_causal_factors'][0].get('importance', 0)
            )
        
        # List top 5 factors
        top5 = dashboard_data.get('top_causal_factors', [])[:5]
        variant_summary["top_5_factors"] = [
            {
                "feature": f['feature'],
                "importance": f.get('causal_responsibility', 
                                  f.get('importance', 
                                       f.get('combined_importance_norm', 0)))
            }
            for f in top5
        ]
    
    # Feature count
    feature_file = CALCULATOR_DIR / "outputs" / "models" / variant_dir / "feature_names.json"
if feature_file.exists():
    with open(feature_file, 'r') as f:
        features = json.load(f)
        summary["model"]["total_features"] = len(features)
        summary["model"]["key_features"] = {
            "primary_etiology": 'primary_etiology' in features or any('primary_etiology' in f for f in features),
            "vad_combined": 'vad_combined' in features,
            "vent_combined": 'vent_combined' in features,
            "ecmo_combined": 'ecmo_combined' in features,
            "donor_weight_ratio": 'donor_weight_ratio' in features,
            "donor_size_ratio": 'donor_size_ratio' in features,
            "chd_lat": 'chd_lat' in features,
            "egfr_tx": 'egfr_tx' in features,
            "txfcpra": 'txfcpra' in features,
            "hxsurg": 'hxsurg' in features
        }
    
    summary["models"][variant_name] = variant_summary

# Save summary
summary_file = CALCULATOR_DIR / "outputs" / "workflow_summary.json"
with open(summary_file, 'w') as f:
    json.dump(summary, f, indent=2)

print(f"\n✓ Workflow summary saved to: {summary_file}")
print("\nSummary:")
print(json.dumps(summary, indent=2))

## 6. Deploy Risk Calculator (Lambda + S3)

Deploy the trained models and dashboard to AWS Lambda and S3 for production use.

### Deployment Overview

**Components:**
- **AWS Lambda**: Container-based function with models baked in
- **API Gateway**: REST API endpoints for risk calculation
- **S3**: Static HTML dashboard hosting

**Prerequisites:**
- AWS CLI configured with appropriate permissions
- Docker installed and running
- Models trained (Section 3)
- SHAP/FFA analysis complete (Section 4)
- Risk distributions computed (if needed)

### Deployment Steps

1. **Prepare Lambda Directory**: Copy models, dashboard data, and risk distributions
2. **Build Docker Image**: Create container image with models
3. **Push to ECR**: Upload image to AWS Elastic Container Registry
4. **Update Lambda**: Deploy new container image to Lambda function
5. **Setup API Gateway**: Configure REST API endpoints
6. **Upload HTML**: Deploy dashboard HTML to S3

In [ ]:
# Step 1: Prepare Lambda Directory (Idempotent - only updates if needed)
import subprocess
from pathlib import Path
import os

print(f"\n{'=' * 80}")
print("Step 1: Preparing Lambda Directory")
print(f"{'=' * 80}")

risk_dashboard_dir = CALCULATOR_DIR / "risk_dashboard"
prepare_script = risk_dashboard_dir / "prepare_lambda_dir_phts.py"
lambda_dir = risk_dashboard_dir / "lambda_dir_phts"

# Check if Lambda directory already exists and is up to date
needs_prepare = True
if lambda_dir.exists():
    # Check if models exist and are recent
    models_dir = lambda_dir / "models" / COHORT
    dashboard_dir = lambda_dir / "dashboard_data" / COHORT
    
    # Check if source models are newer than lambda_dir models
    source_models_dir = CALCULATOR_DIR / "outputs" / "models" / COHORT
    source_dashboard_dir = CALCULATOR_DIR / "outputs" / "shap_ffa" / COHORT
    
    if models_dir.exists() and source_models_dir.exists():
        # Get most recent model file modification time
        source_model_files = list(source_models_dir.glob("*.cbm")) + list(source_models_dir.glob("*.ubj"))
        lambda_model_files = list(models_dir.glob("*.cbm")) + list(models_dir.glob("*.ubj"))
        
        if source_model_files and lambda_model_files:
            source_mtime = max(f.stat().st_mtime for f in source_model_files)
            lambda_mtime = max(f.stat().st_mtime for f in lambda_model_files)
            
            if source_mtime <= lambda_mtime:
                print(f"\n✓ Lambda directory is up to date")
                print(f"  Source models: {len(source_model_files)} files")
                print(f"  Lambda models: {len(lambda_model_files)} files")
                print(f"  Last update: {lambda_mtime}")
                needs_prepare = False

if needs_prepare and prepare_script.exists():
    print(f"\nRunning: {prepare_script}")
    print("This will copy/update models, dashboard data, and risk distributions to lambda_dir_phts/")
    print("-" * 80)
    
    try:
        result = subprocess.run(
            [sys.executable, str(prepare_script)],
            cwd=str(risk_dashboard_dir),
            capture_output=False,
            text=True
        )
        
        if result.returncode == 0:
            print(f"\n✓ Lambda directory prepared successfully!")
            
            # Check what was created/updated
            if lambda_dir.exists():
                print(f"\n  Lambda directory structure:")
                print(f"    {lambda_dir}")
                
                # Check models
                models_dir = lambda_dir / "models" / COHORT
                if models_dir.exists():
                    model_files = list(models_dir.glob("*.cbm")) + list(models_dir.glob("*.ubj"))
                    print(f"    ✓ Models: {len(model_files)} files")
                
                # Check dashboard data
                dashboard_dir = lambda_dir / "dashboard_data" / COHORT
                if dashboard_dir.exists():
                    dashboard_files = list(dashboard_dir.glob("*.json")) + list(dashboard_dir.glob("*.csv"))
                    print(f"    ✓ Dashboard data: {len(dashboard_files)} files")
                
                # Check risk distributions
                risk_dist_dir = lambda_dir / "risk_distributions"
                if risk_dist_dir.exists():
                    risk_files = list(risk_dist_dir.glob("*.json"))
                    print(f"    ✓ Risk distributions: {len(risk_files)} files")
        else:
            print(f"\n⚠ Script exited with code: {result.returncode}")
    except Exception as e:
        print(f"\n✗ Error preparing Lambda directory: {e}")
        logger.error("Error preparing Lambda directory", exc_info=True)
elif not prepare_script.exists():
    print(f"\n⚠ Prepare script not found: {prepare_script}")
    print("  Expected location: risk_dashboard/prepare_lambda_dir_phts.py")

print(f"\n{'=' * 80}")

In [ ]:
# Step 2: Build and Push Docker Image (Idempotent - only if Lambda dir changed)
print(f"\n{'=' * 80}")
print("Step 2: Build and Push Docker Image")
print(f"{'=' * 80}")

docker_script = risk_dashboard_dir / "docker_build_phts.sh"

# Check if Docker image needs to be rebuilt
# Rebuild if lambda_dir was just updated or if image doesn't exist
needs_docker_build = needs_prepare  # Rebuild if we just prepared lambda_dir

if docker_script.exists():
    print(f"\nDocker build strategy:")
    print(f"  - Lambda directory was {'updated' if needs_prepare else 'unchanged'}")
    print(f"  - Docker image will be {'built' if needs_docker_build else 'skipped (use --force to rebuild)'}")
    print("-" * 80)
    print("\nThis will:")
    print("  1. Build Docker image with models and dependencies")
    print("  2. Push image to AWS ECR (Elastic Container Registry)")
    print("-" * 80)
    print("\n⚠ Note: This requires:")
    print("  - Docker installed and running")
    print("  - AWS CLI configured with ECR permissions")
    print("  - AWS credentials with push access to ECR")
    print("-" * 80)
    
    if needs_docker_build:
        response = input("\nProceed with Docker build? (y/n): ").strip().lower()
    else:
        print("\n⏭ Skipping Docker build (Lambda directory unchanged)")
        print("  To force rebuild, run: ./docker_build_phts.sh --force")
        response = 'n'
    
    if response == 'y':
        try:
            result = subprocess.run(
                ["bash", str(docker_script)],
                cwd=str(risk_dashboard_dir),
                capture_output=False,
                text=True
            )
            
            if result.returncode == 0:
                print(f"\n✓ Docker image built and pushed successfully!")
                print(f"\n  Next: Get ECR URI from output above and use it to update Lambda")
            else:
                print(f"\n⚠ Docker build exited with code: {result.returncode}")
        except Exception as e:
            print(f"\n✗ Error building Docker image: {e}")
            logger.error("Error building Docker image", exc_info=True)
else:
    print(f"\n⚠ Docker build script not found: {docker_script}")
    print("  Expected location: risk_dashboard/docker_build_phts.sh")

print(f"\n{'=' * 80}")

In [ ]:
# Step 3: Update Lambda Function (Idempotent - only if Docker image was updated)
print(f"\n{'=' * 80}")
print("Step 3: Update Lambda Function")
print(f"{'=' * 80}")

lambda_function_name = "phts-risk-calculator"
region = "us-east-1"

# Check if Lambda function exists
lambda_exists = False
try:
    result = subprocess.run(
        ["aws", "lambda", "get-function", "--function-name", lambda_function_name, "--region", region],
        capture_output=True,
        text=True,
        timeout=5
    )
    if result.returncode == 0:
        lambda_exists = True
        print(f"\n✓ Lambda function exists: {lambda_function_name}")
except:
    print(f"\n⚠ Could not check Lambda function status")
    print("  (AWS CLI may not be configured)")

if lambda_exists and needs_docker_build:
    print(f"\nLambda function will be updated with new Docker image")
    print("-" * 80)
    print("\n⚠ Note: AWS credentials are automatically used from EC2 instance role")
    print("  (No need to run 'aws configure' if instance has IAM role attached)")
    print("-" * 80)
    
    # Get AWS account ID and ECR URI
    try:
        result = subprocess.run(
            ["aws", "sts", "get-caller-identity", "--query", "Account", "--output", "text"],
            capture_output=True,
            text=True,
            timeout=5
        )
        if result.returncode == 0:
            account_id = result.stdout.strip()
            ecr_uri = f"{account_id}.dkr.ecr.{region}.amazonaws.com/phts-risk-calculator:latest"
            print(f"\n  AWS Account ID: {account_id}")
            print(f"  ECR URI: {ecr_uri}")
            
            # Also get identity to show which role is being used
            identity_result = subprocess.run(
                ["aws", "sts", "get-caller-identity", "--output", "json"],
                capture_output=True,
                text=True,
                timeout=5
            )
            if identity_result.returncode == 0:
                import json
                identity = json.loads(identity_result.stdout)
                if "Arn" in identity:
                    print(f"  Using IAM Role: {identity['Arn']}")
            
            response = input("\nProceed with Lambda update? (y/n): ").strip().lower()
            
            if response == 'y':
                try:
                    result = subprocess.run(
                        [
                            "aws", "lambda", "update-function-code",
                            "--function-name", lambda_function_name,
                            "--image-uri", ecr_uri,
                            "--region", region
                        ],
                        capture_output=False,
                        text=True
                    )
                    
                    if result.returncode == 0:
                        print(f"\n✓ Lambda function updated successfully!")
                        print(f"\n  Waiting for update to complete...")
                        # Wait for function to be ready
                        subprocess.run(
                            ["aws", "lambda", "wait", "function-updated",
                             "--function-name", lambda_function_name,
                             "--region", region],
                            capture_output=True
                        )
                        print(f"  ✓ Lambda function is ready")
                    else:
                        print(f"\n⚠ Lambda update exited with code: {result.returncode}")
                except Exception as e:
                    print(f"\n✗ Error updating Lambda: {e}")
                    logger.error("Error updating Lambda", exc_info=True)
            else:
                print("\n⏭ Skipping Lambda update")
        else:
            print("\n⚠ Could not retrieve AWS Account ID")
    except:
        print("\n⚠ Could not retrieve AWS Account ID - check AWS CLI configuration")
elif not lambda_exists:
    print(f"\n⚠ Lambda function '{lambda_function_name}' does not exist")
    print("  Create it first using AWS Console or CLI")
elif not needs_docker_build:
    print(f"\n⏭ Skipping Lambda update (Docker image unchanged)")
else:
    print("\nTo update Lambda function manually:")
    print("-" * 80)
    print("\n1. Get your AWS Account ID:")
    print("   AWS_ACCOUNT_ID=$(aws sts get-caller-identity --query Account --output text)")
    print("\n2. Construct ECR URI:")
    print("   ECR_URI=\"${AWS_ACCOUNT_ID}.dkr.ecr.us-east-1.amazonaws.com/phts-risk-calculator:latest\"")
    print("\n3. Update Lambda function:")
    print("   aws lambda update-function-code \\")
    print("       --function-name phts-risk-calculator \\")
    print("       --image-uri ${ECR_URI} \\")
    print("       --region us-east-1")

print(f"\n{'=' * 80}")

In [ ]:
# Step 4: Verify API Gateway (Idempotent - assumes already set up)
print(f"\n{'=' * 80}")
print("Step 4: Verify API Gateway")
print(f"{'=' * 80}")

print("\n⚠ Note: API Gateway should already be set up")
print("  This step only verifies the configuration")
print("-" * 80)

# Check if API Gateway exists by trying to list APIs
try:
    result = subprocess.run(
        ["aws", "apigateway", "get-rest-apis", "--query", "items[?name=='phts-risk-calculator-api'].id", "--output", "text"],
        capture_output=True,
        text=True,
        timeout=5
    )
    
    if result.returncode == 0 and result.stdout.strip():
        api_id = result.stdout.strip()
        api_url = f"https://{api_id}.execute-api.us-east-1.amazonaws.com/prod"
        print(f"\n✓ API Gateway found:")
        print(f"  API ID: {api_id}")
        print(f"  API URL: {api_url}")
        
        # Test metadata endpoint
        print(f"\n  Testing /metadata endpoint...")
        try:
            test_result = subprocess.run(
                ["curl", "-s", f"{api_url}/metadata?cohort={COHORT}"],
                capture_output=True,
                text=True,
                timeout=5
            )
            if test_result.returncode == 0:
                print(f"  ✓ API Gateway is responding")
            else:
                print(f"  ⚠ API Gateway may not be responding correctly")
        except:
            print(f"  ⚠ Could not test API endpoint (curl may not be available)")
    else:
        print(f"\n⚠ API Gateway not found or AWS CLI not configured")
        print("  If API Gateway needs to be set up, run:")
        print(f"    cd {risk_dashboard_dir}")
        print(f"    ./setup_api_gateway.sh")
except:
    print(f"\n⚠ Could not verify API Gateway (AWS CLI may not be configured)")
    print("  Assuming API Gateway is already set up")

print(f"\n{'=' * 80}")

In [ ]:
# Step 5: Upload HTML to S3 (Idempotent - only if HTML changed)
print(f"\n{'=' * 80}")
print("Step 5: Upload HTML Dashboard to S3")
print(f"{'=' * 80}")

html_file = risk_dashboard_dir / "phts_dashboard.html"
s3_bucket = "jerome-dixon.io"  # Update if different
s3_prefix = "uva/phts-risk-calculator"
s3_path = f"s3://{s3_bucket}/{s3_prefix}/index.html"

if html_file.exists():
    print(f"\nHTML file found: {html_file}")
    
    # Check if S3 file exists and compare modification times
    needs_upload = True
    try:
        result = subprocess.run(
            ["aws", "s3", "ls", s3_path, "--region", "us-east-1"],
            capture_output=True,
            text=True,
            timeout=5
        )
        
        if result.returncode == 0 and result.stdout.strip():
            # S3 file exists - check if local is newer
            local_mtime = html_file.stat().st_mtime
            
            # Parse S3 last modified time from ls output
            # Format: "2026-01-26 10:30:45    12345 index.html"
            s3_output = result.stdout.strip()
            if s3_output:
                print(f"\n✓ S3 file exists")
                print(f"  Checking if local file is newer...")
                
                # Get S3 file metadata
                head_result = subprocess.run(
                    ["aws", "s3api", "head-object", "--bucket", s3_bucket, 
                     "--key", f"{s3_prefix}/index.html", "--region", "us-east-1"],
                    capture_output=True,
                    text=True,
                    timeout=5
                )
                
                if head_result.returncode == 0:
                    import json
                    s3_meta = json.loads(head_result.stdout)
                    s3_mtime_str = s3_meta.get("LastModified", "")
                    if s3_mtime_str:
                        from datetime import datetime
                        s3_mtime = datetime.fromisoformat(s3_mtime_str.replace("Z", "+00:00")).timestamp()
                        
                        if local_mtime <= s3_mtime:
                            print(f"  ✓ S3 file is up to date (local: {local_mtime}, S3: {s3_mtime})")
                            needs_upload = False
                        else:
                            print(f"  ⚠ Local file is newer - will upload")
        else:
            print(f"\n⚠ S3 file not found - will upload")
    except:
        print(f"\n⚠ Could not check S3 file status (AWS CLI may not be configured)")
        print(f"  Will attempt upload")
    
    if needs_upload:
        print(f"\nUploading to S3:")
        print("-" * 80)
        print(f"  Source: {html_file}")
        print(f"  Destination: {s3_path}")
        print("-" * 80)
        print("\n⚠ Note: This requires:")
        print("  - AWS CLI configured")
        print("  - S3 write permissions")
        print("  - Bucket exists and is accessible")
        print("-" * 80)
        
        response = input("\nProceed with S3 upload? (y/n): ").strip().lower()
        
        if response == 'y':
            try:
                result = subprocess.run(
                    [
                        "aws", "s3", "cp",
                        str(html_file),
                        s3_path,
                        "--content-type", "text/html",
                        "--cache-control", "no-cache",
                        "--region", "us-east-1"
                    ],
                    capture_output=False,
                    text=True
                )
                
                if result.returncode == 0:
                    print(f"\n✓ HTML uploaded successfully to S3!")
                    print(f"\n  Dashboard URL: https://{s3_bucket}/{s3_prefix}/")
                else:
                    print(f"\n⚠ S3 upload exited with code: {result.returncode}")
            except Exception as e:
                print(f"\n✗ Error uploading to S3: {e}")
                logger.error("Error uploading to S3", exc_info=True)
        else:
            print("\n⏭ Skipping S3 upload")
    else:
        print(f"\n⏭ Skipping S3 upload (file is up to date)")
else:
    print(f"\n⚠ HTML file not found: {html_file}")
    print("  Expected location: risk_dashboard/phts_dashboard.html")

print(f"\n{'=' * 80}")

### Deployment Verification

After deployment, verify all components are working:

1. **Lambda Function**: Check CloudWatch logs
2. **API Gateway**: Test endpoints (`/metadata`, `/risk`, `/causal`)
3. **S3 Dashboard**: Load HTML page and test risk calculation
4. **CORS**: Verify browser can call API without CORS errors

### Quick Deployment Script

For automated deployment, use the complete deployment script:

```bash
cd graft-loss/cohort_analysis/calculator/risk_dashboard
./deploy_complete.sh
```

This script automates all deployment steps.

### Documentation

For detailed deployment instructions, see:
- `docs/calculator/README_deployment.md` - Complete deployment guide
- `risk_dashboard/README_DEPLOYMENT.md` - Deployment reference
- `risk_dashboard/README_ARCHITECTURE.md` - Architecture overview

# Final Step: Shutdown EC2 Instance

In [ ]:
# Set SHUTDOWN_EC2 = True to enable, False to disable
SHUTDOWN_EC2 = True  # Change to True to enable auto-shutdown

print(f"\n{'=' * 80}")
print("Final Step: EC2 Instance Shutdown (Optional)")
print(f"{'=' * 80}")

if SHUTDOWN_EC2:
    print("\nShutting down EC2 instance...")
    print("-" * 80)
    
    import subprocess
    import shutil
    import os
    
    # Get instance ID from EC2 metadata service
    try:
        result = subprocess.run(
            ["curl", "-s", "http://169.254.169.254/latest/meta-data/instance-id"],
            capture_output=True,
            text=True,
            timeout=5
        )
        instance_id = result.stdout.strip()
        
        if instance_id and len(instance_id) > 0:
            print(f"Instance ID: {instance_id}")
            
            # Find AWS CLI
            aws_cmd = shutil.which("aws")
            if not aws_cmd:
                # Try common paths
                aws_paths = [
                    "/usr/local/bin/aws",
                    "/usr/bin/aws",
                    "/home/ec2-user/.local/bin/aws"
                ]
                for path in aws_paths:
                    if os.path.exists(path):
                        aws_cmd = path
                        break
            
            if aws_cmd:
                # Stop the instance (use terminate-instances for permanent deletion)
                shutdown_cmd = [aws_cmd, "ec2", "stop-instances", "--instance-ids", instance_id]
                
                print(f"Running: {' '.join(shutdown_cmd)}")
                result = subprocess.run(shutdown_cmd, capture_output=True, text=True)
                
                if result.returncode == 0:
                    print("\n✓ EC2 instance stop command sent successfully")
                    print("Instance will stop in a few moments.")
                    print("Note: This is a STOP (not terminate), so you can restart it later.")
                    logger.info(f"EC2 instance {instance_id} stop command sent successfully")
                else:
                    print(f"\n⚠ EC2 stop command returned exit code {result.returncode}.")
                    print("Check AWS credentials and permissions.")
                    if result.stderr:
                        print(f"Error: {result.stderr}")
                    logger.warning(f"EC2 stop command failed: {result.stderr}")
            else:
                print("\nWarning: AWS CLI not found. Cannot shutdown instance.")
                print("Install AWS CLI or ensure it's in your PATH.")
                logger.warning("AWS CLI not found, cannot shutdown EC2 instance")
        else:
            print("\nWarning: Could not determine instance ID. Skipping shutdown.")
            print("If you want to shutdown manually, use:")
            print("  aws ec2 stop-instances --instance-ids <your-instance-id>")
            logger.warning("Could not determine EC2 instance ID")
    except subprocess.TimeoutExpired:
        print("\nWarning: Timeout retrieving instance ID from metadata service.")
        print("If running on EC2, check that metadata service is accessible.")
        logger.warning("Timeout retrieving EC2 instance ID from metadata service")
    except Exception as e:
        print(f"\nWarning: Could not retrieve instance ID: {e}")
        print("If you want to shutdown manually, use:")
        print("  aws ec2 stop-instances --instance-ids <your-instance-id>")
        logger.warning(f"Error retrieving EC2 instance ID: {e}")
else:
    print("\nEC2 Auto-Shutdown: DISABLED")
    print("To enable auto-shutdown, set SHUTDOWN_EC2 = True in this cell.")
    print("Instance will continue running.")

print(f"\n{'=' * 80}")
print("Workflow Complete!")
print(f"{'=' * 80}")